# 01 - Compare our QUANT results with the paper

Compares the output of the authors' unmodified `run.py` (our run) with the published results in `original/results/`.

- **Now:** smoke test (`results/results.csv`, 5 datasets x 3 resamples).
- **Later:** switch `OUR_RESULTS` to `results/original_full_run1/results.csv` (142 datasets x 30 resamples).

Questions:
1. Do our accuracies match the published ones, per dataset and resample?
2. Are the differences within the expected noise (unseeded trees + resample variance), or systematic?
3. Does the overall mean match the paper (0.8568 over 142 datasets)?

## 1. Settings

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

REPO = Path.cwd().parent            # the notebook runs from quant-reproduction/notebooks
OUR_RESULTS = REPO / "results" / "results.csv"                            # smoke test
# OUR_RESULTS = REPO / "results" / "original_full_run1" / "results.csv"   # full run (when finished)

PUBLISHED_RESAMPLES = REPO / "original" / "results" / "accuracy_resamples_ucr142.csv"
DATA_TS = REPO / "data" / "ts"
PAPER_MEAN_ACCURACY = 0.8568        # Fig. 6 of the paper

## 2. Load the results in the authors' (wide) format

The published file is *wide*: one row per dataset, one column per resample (`0`..`29`).
`run.py` writes a *long* file (one row per dataset + resample), so we pivot ours into the same wide layout.
Then both tables line up cell by cell: same dataset (row), same resample (column).

In [ ]:
ours_long = pd.read_csv(OUR_RESULTS)

ours = ours_long.pivot(index="dataset", columns="resample", values="accuracy")
ours.columns = ours.columns.astype(str)          # "0", "1", ... like the published file
ours

In [ ]:
published_all = pd.read_csv(PUBLISHED_RESAMPLES, index_col=0)   # all 142 datasets x 30 resamples

# only the datasets and resamples that we ran, in the same order as ours
published = published_all.loc[ours.index, ours.columns]
published

## 3. Differences (ours - published)

Because both tables have the same rows and columns, the difference is a plain subtraction.

In [ ]:
diff = ours - published
diff.round(4)

### Differences in test cases

An accuracy difference of 0.01 means something different for a test set of 30 cases than for one of 1000.
Converting the difference to a *number of test cases* makes it easier to read.
Resamples keep the original train/test sizes, so we count the cases in the `_TEST.ts` file.

In [ ]:
def count_test_cases(name):
    """Number of series in <name>_TEST.ts = number of non-empty lines after the @data line."""
    lines = (DATA_TS / name / f"{name}_TEST.ts").read_text(encoding="utf-8").splitlines()
    start = [i for i, line in enumerate(lines) if line.strip().lower() == "@data"][0]
    return sum(1 for line in lines[start + 1:] if line.strip())

n_test = pd.Series({name: count_test_cases(name) for name in ours.index})

diff_cases = diff.mul(n_test, axis=0).round()    # each row multiplied by that dataset's test size
diff_cases.insert(0, "n_test", n_test)
diff_cases

## 4. Per dataset: is the difference within the expected noise?

For each dataset we compare the mean of our resamples with the mean of the *same* published resamples.

The published 30 resamples tell us how much accuracy varies from split to split (`std`).
The standard error of a mean over *k* resamples is `std / sqrt(k)`; for a difference of two such means it is about
`sqrt(2) * std / sqrt(k)`. We use this as a rough z-score: |z| < 2 means the difference is within the noise.

(This is a rough check that mixes split noise with seed noise. The planned 10-seed experiment will measure seed noise on its own.)

In [ ]:
per_dataset = pd.DataFrame({
    "k": ours.notna().sum(axis=1),                     # number of resamples we ran
    "ours_mean": ours.mean(axis=1),
    "published_mean": published.mean(axis=1),         # same resamples as ours
    "published_std": published_all.loc[ours.index].std(axis=1),   # spread over all 30 resamples
})
per_dataset["diff"] = per_dataset["ours_mean"] - per_dataset["published_mean"]
per_dataset["se_diff"] = np.sqrt(2) * per_dataset["published_std"] / np.sqrt(per_dataset["k"])
per_dataset["z"] = per_dataset["diff"] / per_dataset["se_diff"].replace(0, np.nan)
per_dataset["within_noise"] = per_dataset["z"].abs().fillna(0) < 2
per_dataset.round(4)

## 5. Overall comparison

In [ ]:
n_runs = ours.notna().sum().sum()
print(f"datasets compared:           {len(ours)}")
print(f"runs compared:               {n_runs}")
print(f"mean accuracy (ours):        {ours.stack().mean():.4f}")
print(f"mean accuracy (published):   {published.stack().mean():.4f}   (same datasets and resamples)")
print(f"mean difference:             {diff.stack().mean():+.4f}")
print(f"exact matches:               {(diff.abs() < 1e-6).sum().sum()} of {n_runs}")
print(f"datasets within noise:       {per_dataset['within_noise'].sum()} of {len(per_dataset)}")
print(f"paper mean (142 x 30):       {PAPER_MEAN_ACCURACY}")

### Wilcoxon signed-rank test

Tests whether our per-dataset means are *systematically* higher or lower than the published ones
(p < 0.05 would suggest a systematic difference). With only 5 datasets it has almost no power;
it becomes meaningful with the full 142 datasets.

In [ ]:
d = per_dataset["diff"]
d = d[d.abs() > 1e-9]          # Wilcoxon ignores exact ties (zero differences)
if len(d) >= 5:
    print(stats.wilcoxon(d))
else:
    print(f"only {len(d)} non-zero differences - too few for a Wilcoxon test")

## 6. Plots

For plotting, `.stack()` turns each wide table into one long list of values (dataset + resample pairs).

In [ ]:
x = published.stack()          # published accuracy per dataset + resample
y = ours.stack()               # our accuracy, same order
d_all = diff.stack()

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

# Left: ours vs published. On the diagonal = identical.
ax = axes[0]
ax.scatter(x, y, alpha=0.7)
lo = min(x.min(), y.min()) - 0.02
ax.plot([lo, 1], [lo, 1], "k--", linewidth=1)
ax.set_xlabel("published accuracy")
ax.set_ylabel("our accuracy")
ax.set_title("Per dataset + resample")

# Right: Bland-Altman style - difference vs average accuracy.
ax = axes[1]
ax.scatter((x + y) / 2, d_all, alpha=0.7)
ax.axhline(0, color="k", linewidth=1)
ax.axhline(d_all.mean(), color="C1", linestyle="--", label=f"mean diff {d_all.mean():+.4f}")
ax.set_xlabel("average accuracy")
ax.set_ylabel("ours - published")
ax.set_title("Difference vs accuracy")
ax.legend()
plt.tight_layout()
plt.show()

## 7. TODO once the full run is finished

- [ ] Switch `OUR_RESULTS` to the full run and re-run all cells.
- [ ] Check the overall mean against 0.8568 (all 142 datasets x 30 resamples).
- [ ] Mean over the 112 original datasets vs the paper's 0.8671 (Fig. 1).
- [ ] Look at resample 0 separately (the original UCR split, identical for everyone).
- [ ] List the datasets with |z| >= 2 and look for a pattern (e.g. the 4 corrected datasets Covid3MonthDiscrete, FloodModeling1-3Discrete).
- [ ] Add the bake-off redux QUANT results as a third comparison.
- [ ] Save the comparison table to `results/` for the README.